# VLM 배치 프롬프트 방식 검증 — 프로덕션 SAM3 탐지 + 배치 하자 설명

**목적**: 프로덕션 탐지 파이프라인(`worker/models/sam3/`)은 그대로 재사용하고, 하자
설명(VLM) 생성 방식만 프로덕션의 크롭 개별 호출(`quality_vlm.describer.attach_descriptions`)
대신 **원본 이미지 1장 + 번호별 위치·유형 텍스트 힌트로 한 번에 묻는 배치 방식**
(`AI/validation/QUAL/vlm-description-check/loaders.py`의 `build_batch_prompt`와 동일 문구,
2026-08-10 사용자가 최종 확정한 프롬프트)으로 바꿔서 결과를 비교한다.

탐지·항목화(`extract_findings`)는 **프로덕션 코드를 그대로 import**해서 쓴다 — 로직을
복제하지 않는다(`worker/validation/quality_check_e2e_validation.ipynb`와 같은 원칙).
배치 프롬프트 빌더만 이 노트북 전용이다 — 배치 방식 자체가 아직 프로덕션에 없어서다.

## 실행 방법

- **반드시 `worker/` 또는 `worker/validation/`에서 연다**(sys.path가 워커 스크립트와 같아야 함).
- 워커 실행 때와 같은 conda 환경(예: `qual`)에서 커널을 띄운다.
- 기본으로 `C:\Users\SSAFY\Downloads\testt1` 폴더의 이미지 전부를 대상으로 한다 —
  다른 폴더를 쓰려면 환경변수 `QUALITY_TEST_IMAGE_DIR`을 그 경로로 지정하거나 아래
  설정 셀에서 `IMAGE_DIR`을 직접 바꾼다.
- `QUALITY_BACKEND`/`QUALITY_VLM_BACKEND` 기본값은 각각 `sam3_pcs`/`qwen3_vl`이다(실제
  탐지·VLM 비교가 목적이라 mock이 아니라 이걸 기본으로 뒀다) — 필요하면 환경변수로 덮어쓴다.

실행 순서: 0(설정) → 1(세그멘터 로드+탐지) → 2(항목화) → 3(번호 박스 시각화) →
4(VLM 로드) → 5(이미지별 배치 프롬프트 호출+응답 확인).

In [ ]:
import os
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import ImageDraw, ImageFont

%matplotlib inline

# quality_check_worker_example.py와 완전히 같은 sys.path 설정 — 같은 모듈을 같은 방식
# (`sam3.xxx`/`quality_vlm.xxx` 네임스페이스)으로 import하기 위함.
_CWD = Path(os.getcwd())
_WORKER_DIR = _CWD if (_CWD / "models").is_dir() else _CWD.parent
_MODELS_DIR = str(_WORKER_DIR / "models")
assert os.path.isdir(_MODELS_DIR), (
    f"'{_MODELS_DIR}'가 없습니다 — 이 노트북은 worker/ 또는 worker/validation/에서 열어야 합니다."
)
sys.path.insert(0, os.getenv("QUALITY_MODEL_CODE_PATH", _MODELS_DIR))
sys.path.insert(0, str(_WORKER_DIR))

from quality_vlm import registry as vlm_registry  # noqa: E402
from quality_vlm.loader import generate_from_image, load_qwen_vl  # noqa: E402
from quality_vlm.prompts import _FEW_SHOT_EXAMPLES  # noqa: E402
from sam3 import registry  # noqa: E402
from sam3.contracts import BaseColor, GarmentCategory  # noqa: E402
from sam3.findings import extract_findings  # noqa: E402
from sam3.prompts import SECTION_LABELS_KO, select_prompt_set  # noqa: E402
from sam3.segmenter import InspectionTarget, build_segmenter  # noqa: E402

print("production 모듈 import 완료 — 탐지/항목화는 워커와 동일 코드 경로")

## 0. 설정 — 백엔드 + 테스트 이미지 폴더

In [ ]:
# 실제 탐지/VLM 비교가 목적이라 mock이 아니라 실백엔드를 기본값으로 둔다.
# 다른 값으로 테스트하고 싶으면 환경변수로 덮어쓰거나 이 셀에서 직접 바꾼다.
QUALITY_BACKEND = os.getenv("QUALITY_BACKEND", "sam3_pcs")
QUALITY_VLM_BACKEND = os.getenv("QUALITY_VLM_BACKEND", "qwen3_vl")

# 이 폴더에 있는 이미지 전부를 한 상품의 사진들로 취급해서 검사한다.
IMAGE_DIR = Path(os.getenv("QUALITY_TEST_IMAGE_DIR", r"C:\Users\SSAFY\Downloads\testt1"))
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp"}
IMAGE_PATHS = sorted(p for p in IMAGE_DIR.glob("*") if p.suffix.lower() in IMAGE_EXTS)
assert IMAGE_PATHS, f"'{IMAGE_DIR}'에 이미지가 없습니다."

CATEGORY = GarmentCategory.top_shirt
BASE_COLOR = BaseColor.light

print(f"QUALITY_BACKEND     = {QUALITY_BACKEND}")
print(f"QUALITY_VLM_BACKEND = {QUALITY_VLM_BACKEND}")
print(f"IMAGE_DIR           = {IMAGE_DIR}")
print(f"대상 이미지 {len(IMAGE_PATHS)}장: {[p.name for p in IMAGE_PATHS]}")

## 1. 세그멘터 로드 + 탐지 (프로덕션 SAM3 코드 그대로)

In [ ]:
_t0 = time.perf_counter()
segmenter_config = registry.segmenter_config(QUALITY_BACKEND)
segmenter = build_segmenter(QUALITY_BACKEND)
print(f"[로드] 세그멘터 {time.perf_counter() - _t0:.1f}s, model_id={segmenter_config['model_id']}")

prompt_set = select_prompt_set(CATEGORY, BASE_COLOR)
target = InspectionTarget(
    product_id=999002,  # 테스트 전용 임의값
    image_urls=tuple(str(p) for p in IMAGE_PATHS),
    prompt_set=prompt_set,
)

results = []
for i, path in enumerate(IMAGE_PATHS):
    _t0 = time.perf_counter()
    result = segmenter.segment(target, i)
    print(
        f"  image[{i}] {path.name:<30s} {time.perf_counter() - _t0:5.1f}s  "
        f"instances={len(result.instances)}"
    )
    results.append(result)

## 2. 하자 항목화 (extract_findings, 프로덕션 코드 그대로)

In [ ]:
findings_result = extract_findings(results, prompt_set, segmenter_config)
print(f"findings={len(findings_result.findings)}")
for f in findings_result.findings:
    print(
        f"  - image[{f.image_index}] no={f.no} {f.label} "
        f"section={SECTION_LABELS_KO[f.section]} status={f.status.value} conf={f.confidence:.3f}"
    )

## 3. 번호 박스 시각화 (배치 프롬프트가 참조하는 번호와 실제 위치 대조용)

In [ ]:
def draw_numbered_boxes(image, image_findings):
    """finding.bbox(정규화 xyxy)에 번호만 그린다 — 실제 응답 문구와 대조해서 보기 위한
    용도라 프로덕션 오버레이(annotated_images)만큼 정교할 필요는 없다.
    """
    canvas = image.convert("RGB").copy()
    draw = ImageDraw.Draw(canvas)
    w, h = canvas.size
    font = ImageFont.load_default()
    for f in image_findings:
        x1, y1, x2, y2 = f.bbox
        box = (x1 * w, y1 * h, x2 * w, y2 * h)
        draw.rectangle(box, outline="red", width=4)
        tag_x, tag_y = box[0], max(0, box[1] - 24)
        draw.rectangle([tag_x, tag_y, tag_x + 24, tag_y + 24], fill="red")
        draw.text((tag_x + 6, tag_y + 4), str(f.no), fill="white", font=font)
    return canvas


findings_by_image = {}
for f in findings_result.findings:
    findings_by_image.setdefault(f.image_index, []).append(f)

_with_findings = [i for i in range(len(results)) if findings_by_image.get(i)]
if _with_findings:
    fig, axes = plt.subplots(1, len(_with_findings), figsize=(6 * len(_with_findings), 6), squeeze=False)
    for ax, i in zip(axes[0], _with_findings):
        ax.imshow(draw_numbered_boxes(results[i].base_image, findings_by_image[i]))
        ax.set_title(f"image[{i}] {len(findings_by_image[i])}건")
        ax.axis("off")
    plt.show()
else:
    print("하자가 검출된 이미지가 없습니다.")

## 4. VLM 로드 (프로덕션 `quality_vlm.loader` 그대로)

In [ ]:
VLM_CONFIG = vlm_registry.vlm_config(QUALITY_VLM_BACKEND)
print(f"VLM model_id={VLM_CONFIG['model_id']}, quantization={VLM_CONFIG['quantization']}")

_t0 = time.perf_counter()
vlm_bundle = load_qwen_vl(VLM_CONFIG["model_id"], quantization=VLM_CONFIG["quantization"])
print(f"[로드] VLM {time.perf_counter() - _t0:.1f}s")

## 5. 이미지별 배치 프롬프트 호출

**2026-08-10 사용자가 확정한 프롬프트**를 그대로 쓴다 — few-shot 예시만 프로덕션
`quality_vlm.prompts._FEW_SHOT_EXAMPLES`(hotfix/472로 "원단" 단어를 뺀 최신 버전)를
재사용하고, 나머지 배치 전용 문구(번호별 위치·유형 목록 + 지시문)는 이 노트북에서
그대로 조립한다.

In [ ]:
_TOKENS_PER_FINDING = 80  # 크롭 방식 max_new_tokens 기본값과 같은 예산(1건당)


def build_batch_prompt(image_findings):
    """원본 이미지 1장 + 번호별 위치·유형 텍스트 힌트로 모든 finding을 한 번에 묻는다.

    2026-08-10 사용자가 최종 확정한 문구를 그대로 옮겼다(원본:
    `AI/validation/QUAL/vlm-description-check/loaders.py`의 `build_batch_prompt`).
    """
    examples = "\n".join(f"- {ex}" for ex in _FEW_SHOT_EXAMPLES)
    listing = "\n".join(
        f"{f.no}번 ({SECTION_LABELS_KO[f.section]} 부근): '{f.label}' 하자"
        for f in image_findings
    )
    return (
        "아래는 하자 시각 설명의 좋은 답변 예시입니다.\n"
        f"{examples}\n\n"
        "이 이미지는 중고 의류 사진입니다. 번호별 하자의 대략적인 위치와 유형을 미리 "
        f"알려드립니다(위치는 참고만 하세요):\n{listing}\n\n"
        "각 번호에 대해 '번호. 설명' 형식으로 한 줄씩, 위 예시처럼 그 위치에서 가장 눈에 "
        "띄는 시각적 특징 딱 한 가지만 짧게 한국어 한 문장으로 설명해 주세요. 색상, 모양, "
        "크기, 질감을 전부 나열하지 마세요 — 예시들도 그중 하나만 말합니다. 위치는 이미"
        " 알려드렸으니 답변에서 다시 언급하지 마세요. 보이지 않는 내용은 지어내지 마세요."
    )


for image_index in sorted(findings_by_image):
    image_findings = sorted(findings_by_image[image_index], key=lambda f: f.no)
    prompt_text = build_batch_prompt(image_findings)
    max_new_tokens = max(1, len(image_findings)) * _TOKENS_PER_FINDING

    print(f"\n{'=' * 70}\nimage[{image_index}] {IMAGE_PATHS[image_index].name} — {len(image_findings)}건\n{'=' * 70}")
    print("--- 실제 프롬프트 ---")
    print(prompt_text)

    _t0 = time.perf_counter()
    raw_response = generate_from_image(
        vlm_bundle, results[image_index].base_image, prompt_text, max_new_tokens
    )
    print(f"\n--- 응답 ({time.perf_counter() - _t0:.1f}s) ---")
    print(raw_response)

## 결론 (실행 후 채울 것)

- 배치 방식 응답이 소재/원단/부위 단어를 언급하는가(크롭 개별 방식과 비교):
- 번호별 설명이 서로 다른가, 아니면 특정 번호로 쏠리는가:
- finding 수가 많을 때(batch_pipeline_check.ipynb가 지적한 한계 — 8건 이상에서 반복·조기중단)
  재현되는가:
- 채택한다면 `docs/plans/AI/`에 배치 방식 프로덕션 반영 설계 계획부터 쓴다(CH011 선례) —
  이 노트북 코드를 그대로 승격하지 않는다.